# Oil Well Profit & Risk Analysis
## OilyGiant — Seleção de região para novos poços

**Objetivo de negócio.** A OilyGiant precisa decidir em qual de três regiões abrir 200 novos poços de petróleo. Para cada região dispomos de uma amostra de 100.000 poços com três características geológicas (`f0`, `f1`, `f2`) e o volume de reservas (`product`, em milhares de barris).

O fluxo do projeto é:

1. Carregar e preparar os dados das três regiões.
2. Treinar um modelo de **regressão linear** para prever o volume de reservas (condição obrigatória do projeto).
3. Selecionar, em cada região, os **200 poços com maior volume previsto** entre 500 estudados.
4. Estimar lucro e risco via **bootstrapping** (1.000 amostras).
5. Recomendar a região com **risco de prejuízo abaixo de 2,5%** e o **maior lucro médio**.

**Parâmetros comerciais**

| Parâmetro | Valor |
|---|---|
| Pontos estudados por região | 500 |
| Poços selecionados | 200 |
| Orçamento | 100 milhões USD |
| Receita por unidade de produto (mil barris) | 4.500 USD |
| Investimento por poço | 500.000 USD |

**Reprodução de 15/09/2026:** seleção posicional corrigida; Região 1 com lucro médio simulado de US$ 4,61 milhões e risco de 0,7%. Veja [a nota de validação](../VALIDATION.md).

In [1]:
from pathlib import Path

# Works when the kernel starts at the repository root or inside notebooks/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / 'data').is_dir() and (p / 'notebooks').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository root or notebooks/ directory.')
DATA_DIR = PROJECT_ROOT / 'data'


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 12345
state = np.random.RandomState(RANDOM_STATE)

# Condições de negócio
N_STUDY = 500            # pontos estudados por região
N_BEST = 200             # melhores poços selecionados
BUDGET = 100_000_000     # orçamento total (USD)
REVENUE_PER_UNIT = 4500  # receita por unidade de produto (mil barris)
BOOTSTRAP_REPS = 1000

## Etapa 1 — Carregamento e preparação dos dados

Lemos os três arquivos e fazemos uma inspeção rápida: formato, valores ausentes, duplicatas de `id` e volume médio de reservas.

In [3]:
paths = {
    0: DATA_DIR / 'geo_data_0.csv',
    1: DATA_DIR / 'geo_data_1.csv',
    2: DATA_DIR / 'geo_data_2.csv',
}

datasets = {}
for r, p in paths.items():
    datasets[r] = pd.read_csv(p)

datasets[0].head()

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


In [4]:
for r in range(3):
    df = datasets[r]
    print(f"--- Região {r} ---")
    print(f"Formato: {df.shape}")
    print(f"Duplicados em 'id': {df['id'].duplicated().sum()}")
    print(f"Valores ausentes: {df.isna().sum().sum()}")
    print(f"Volume médio de reservas: {df['product'].mean():.4f}\n")

--- Região 0 ---
Formato: (100000, 5)
Duplicados em 'id': 10
Valores ausentes: 0
Volume médio de reservas: 92.5000

--- Região 1 ---
Formato: (100000, 5)
Duplicados em 'id': 4
Valores ausentes: 0
Volume médio de reservas: 68.8250

--- Região 2 ---
Formato: (100000, 5)
Duplicados em 'id': 4
Valores ausentes: 0
Volume médio de reservas: 95.0000



**Observações da preparação.**

- As três bases têm 100.000 linhas e nenhuma com valores ausentes.
- Há um pequeno número de `id` duplicados (10 na região 0, 4 nas regiões 1 e 2). Os registros são mantidos nesta reprodução. Excluir o identificador das variáveis não elimina eventual dependência entre amostras: é necessário investigar se os IDs repetidos representam o mesmo poço e, nesse caso, validar por grupos.
- As características `f0`, `f1`, `f2` já são numéricas. A regressão linear não exige escalonamento para produzir predições corretas; como usaremos apenas regressão linear e nenhum método sensível à escala, não escalamos as características.
- A coluna `id` é apenas um identificador e a removemos das características; `product` é o alvo.

## Etapa 2 — Treinamento e teste do modelo por região

Encapsulamos os passos 2.1 a 2.5 numa função para evitar duplicação de código e aplicá-la às três regiões:

1. Dividir em treino/validação na proporção 75:25.
2. Treinar a regressão linear e prever no conjunto de validação.
3. Guardar predições e respostas corretas.
4. Imprimir o volume médio previsto e o **REQM (RMSE)**.

In [5]:
def train_and_evaluate(df, region):
    features = df.drop(['id', 'product'], axis=1)
    target = df['product']

    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=RANDOM_STATE)

    model = LinearRegression()
    model.fit(features_train, target_train)
    predictions = pd.Series(model.predict(features_valid), index=target_valid.index)

    rmse = mean_squared_error(target_valid, predictions) ** 0.5
    mean_pred = predictions.mean()

    print(f"--- Região {region} ---")
    print(f"Volume médio previsto de reservas: {mean_pred:.4f} mil barris")
    print(f"REQM (RMSE): {rmse:.4f}\n")

    return target_valid.reset_index(drop=True), predictions.reset_index(drop=True), mean_pred, rmse


results = {}
for r in range(3):
    tv, pr, mp, rmse = train_and_evaluate(datasets[r], r)
    results[r] = {'target': tv, 'preds': pr, 'mean_pred': mp, 'rmse': rmse}

--- Região 0 ---
Volume médio previsto de reservas: 92.5926 mil barris
REQM (RMSE): 37.5794

--- Região 1 ---
Volume médio previsto de reservas: 68.7285 mil barris
REQM (RMSE): 0.8931

--- Região 2 ---
Volume médio previsto de reservas: 94.9650 mil barris
REQM (RMSE): 40.0297



**Análise dos resultados do modelo.**

- **Região 1** tem um REQM extraordinariamente baixo (≈ 0,89): as características são quase perfeitamente lineares com o alvo, então o modelo prevê o volume com altíssima precisão. Em contrapartida, seu volume médio previsto (≈ 68,7) é o mais baixo das três.
- **Regiões 0 e 2** têm volumes médios previstos mais altos (≈ 92,6 e ≈ 95,0), mas REQM muito maiores (≈ 37,6 e ≈ 40,0). O modelo nelas é muito menos confiável; muita variância nas predições.

Essa diferença de incerteza é decisiva: na região 1, conseguimos identificar com segurança quais poços são bons; nas outras, a seleção dos "melhores" carrega ruído considerável.

## Etapa 3 — Preparação para o cálculo de lucro

Guardamos os valores-chave e calculamos o **volume mínimo por poço para evitar prejuízo** (break-even): o investimento por poço dividido pela receita por unidade.

In [6]:
break_even_per_well = BUDGET / N_BEST / REVENUE_PER_UNIT

print(f"Investimento por poço: {BUDGET/N_BEST:,.2f} USD")
print(f"Receita por unidade de produto: {REVENUE_PER_UNIT} USD")
print(f"Volume mínimo por poço para evitar prejuízo: {break_even_per_well:.2f} mil barris\n")

print("Comparação com o volume médio de cada região:")
for r in range(3):
    m = datasets[r]['product'].mean()
    status = 'ACIMA' if m > break_even_per_well else 'ABAIXO'
    print(f"  Região {r}: média = {m:.2f} | break-even = {break_even_per_well:.2f} -> {status}")

Investimento por poço: 500,000.00 USD
Receita por unidade de produto: 4500 USD
Volume mínimo por poço para evitar prejuízo: 111.11 mil barris

Comparação com o volume médio de cada região:
  Região 0: média = 92.50 | break-even = 111.11 -> ABAIXO
  Região 1: média = 68.83 | break-even = 111.11 -> ABAIXO
  Região 2: média = 95.00 | break-even = 111.11 -> ABAIXO


**Conclusões da preparação.**

O ponto de equilíbrio é de **111,11 mil barris por poço**. O volume médio das três regiões (92,5; 68,8; 95,0) está **abaixo** desse limiar.

Isso **não** significa que o projeto seja inviável: a estratégia não é perfurar poços aleatórios, e sim selecionar os **200 melhores entre 500 estudados**. Os poços de topo de cada região superam folgadamente a média, e é sobre eles que o lucro é calculado. A análise por bootstrapping a seguir mede exatamente isso.

## Etapa 4 — Função de lucro e lucro dos 200 melhores poços

A função `profit` recebe as respostas reais, as predições e o número de poços a selecionar. Ela ordena pelos **maiores volumes previstos**, soma o volume real correspondente, converte em receita e subtrai o orçamento.

In [7]:
def profit(target, predictions, count):
    # Bootstrap samples can repeat labels. Rank positions to avoid multiplying
    # repeated labels during pandas alignment.
    actual = pd.Series(np.asarray(target))
    predicted = pd.Series(np.asarray(predictions))
    if len(actual) != len(predicted):
        raise ValueError('Targets and predictions must have equal lengths.')
    if not 0 < count <= len(actual):
        raise ValueError('count must fit within the sample.')
    selected_positions = predicted.sort_values(ascending=False).iloc[:count].index
    return actual.iloc[selected_positions].sum() * REVENUE_PER_UNIT - BUDGET


for r in range(3):
    p = profit(results[r]['target'], results[r]['preds'], N_BEST)
    print(f'Região {r}: lucro dos 200 melhores = {p:,.2f} USD')

Região 0: lucro dos 200 melhores = 33,208,260.43 USD
Região 1: lucro dos 200 melhores = 24,150,866.97 USD
Região 2: lucro dos 200 melhores = 27,103,499.64 USD


**Leitura da Etapa 4.** Usando todo o conjunto de validação, a região 0 apresenta o maior lucro pontual (≈ 33,2 mi), seguida da região 2 (≈ 27,1 mi) e da região 1 (≈ 24,2 mi).

Atenção: esse cálculo usa o conjunto de validação inteiro de uma só vez e **não reflete o processo real** (estudar 500 pontos e escolher 200) nem o risco. Por isso ele serve apenas como sanidade — a decisão se baseia no bootstrapping da próxima etapa.

## Etapa 5 — Riscos e lucro por bootstrapping

Para cada região, repetimos 1.000 vezes: sorteamos com reposição **500 poços** (o estudo de campo), aplicamos o modelo escolhendo os **200 melhores** e calculamos o lucro. Disso extraímos:

- **lucro médio**,
- **intervalo central de 95% da distribuição de lucros simulados** (quantis 2,5% e 97,5%),
- **risco de prejuízo** = probabilidade de lucro negativo.

In [8]:
state = np.random.RandomState(RANDOM_STATE)  # repeatable bootstrap reruns

summary = {}
for r in range(3):
    target = results[r]['target']
    preds = results[r]['preds']

    values = []
    for i in range(BOOTSTRAP_REPS):
        target_subsample = target.sample(n=N_STUDY, replace=True, random_state=state)
        preds_subsample = preds[target_subsample.index]
        values.append(profit(target_subsample, preds_subsample, N_BEST))

    values = pd.Series(values)
    summary[r] = {
        'mean': values.mean(),
        'lower': values.quantile(0.025),
        'upper': values.quantile(0.975),
        'risk': (values < 0).mean() * 100,
    }

    print(f"--- Região {r} ---")
    print(f"Lucro médio: {summary[r]['mean']:,.2f} USD")
    print(f"Intervalo bootstrap 95%: ({summary[r]['lower']:,.2f} ; {summary[r]['upper']:,.2f}) USD")
    print(f"Risco de prejuízo: {summary[r]['risk']:.2f}%\n")

--- Região 0 ---
Lucro médio: 3,961,649.85 USD
Intervalo bootstrap 95%: (-1,112,155.46 ; 9,097,669.42) USD
Risco de prejuízo: 6.90%

--- Região 1 ---
Lucro médio: 4,611,558.17 USD
Intervalo bootstrap 95%: (780,508.11 ; 8,629,520.60) USD
Risco de prejuízo: 0.70%

--- Região 2 ---
Lucro médio: 3,929,504.75 USD
Intervalo bootstrap 95%: (-1,122,276.25 ; 9,345,629.15) USD
Risco de prejuízo: 6.50%



In [9]:
# Tabela-resumo comparativa
tabela = pd.DataFrame(summary).T
tabela.index.name = 'regiao'
tabela.columns = ['lucro_medio', 'ic_inferior', 'ic_superior', 'risco_%']
tabela.round(2)

,lucro_medio,ic_inferior,ic_superior,risco_%
regiao,,,,
0,3961649.85,-1112155.46,9097669.42,6.9
1,4611558.17,780508.11,8629520.60,0.7
2,3929504.75,-1122276.25,9345629.15,6.5


## Conclusão final e recomendação

Resultados recalculados em 15/09/2026, após corrigir a seleção por posição no bootstrap:

| Região | Lucro médio | Intervalo bootstrap central de 95% | Risco de prejuízo |
|---|---:|---:|---:|
| 0 | USD 3.96M | USD -1.11M to USD 9.10M | 6.9% |
| 1 | USD 4.61M | USD 0.78M to USD 8.63M | 0.7% |
| 2 | USD 3.93M | USD -1.12M to USD 9.35M | 6.5% |

**Região 1** permanece como a única elegível pelo limite de risco inferior a 2,5%, com lucro médio simulado de **US$ 4,61 milhões** e risco de **0,7%**. As regiões 0 e 2 apresentam riscos de 6,9% e 6,5%.

A reprodução do código anterior retornou os mesmos valores históricos, incluindo US$ 5,18 milhões e 0,3% para a Região 1. Esses números foram substituídos porque o alinhamento por rótulos duplicados multiplicava observações em amostras com reposição.

O intervalo apresentado corresponde aos quantis dos lucros simulados, não a um intervalo de confiança para o lucro médio. A simulação mantém o modelo e a amostra de validação fixos: ela não quantifica toda a incerteza de treinamento nem mudanças nos preços, custos ou condições geológicas. IDs repetidos ainda precisam de investigação antes de uma decisão real. Os resultados são de um estudo educacional, sem retorno financeiro realizado.
